# pwn-scenarios quickstart

Clone the repo, reassemble the dataset from its committed chunks, build the queryable views, and run a few example queries -- no local setup needed.

Repo: https://github.com/AlaBYahya/pwn-scenarios

In [ ]:
!git clone --depth 1 https://github.com/AlaBYahya/pwn-scenarios.git
%cd pwn-scenarios
!pip install -q -r requirements.txt

In [ ]:
# Reassemble the canonical scenarios.jsonl from the git-friendly chunks
!cat data/scenarios/chunks/scenarios.part*.jsonl > data/scenarios/scenarios.jsonl
!wc -l data/scenarios/scenarios.jsonl

In [ ]:
# Build the per-class split and the SQLite index (both gitignored, built locally)
%cd scripts
!python3 build_views.py

In [ ]:
# Example queries via the bundled CLI
!python3 query.py --cwe CWE-89
!python3 query.py --playbook ssrf --severity high
!python3 query.py --search "cache poisoning"

In [ ]:
# Load it directly as a pandas DataFrame if you'd rather work with it that way
import json
import pandas as pd

records = [json.loads(line) for line in open("../data/scenarios/scenarios.jsonl")]
df = pd.json_normalize(records)
df[["vulnerability.class", "vulnerability.cwe", "source.platform"]].head()

## The attack decision graph

`data/graph/attack_graph.json` chains vulnerability classes (including 17 real, NVD-verified CVE chains) into a graph of states and actions. See [docs/GRAPH.md](https://github.com/AlaBYahya/pwn-scenarios/blob/main/docs/GRAPH.md) for the full design.

In [ ]:
!python3 query_graph.py --from web_target_identified
!python3 query_graph.py --best-path --from ssrf_confirmed --to full_cloud_account_compromise